# Fraud Detection System - Data Exploration

This notebook explores the credit card fraud detection dataset and provides initial insights.

## Dataset Information
- **Source**: Kaggle Credit Card Fraud Detection
- **Transactions**: 284,807 credit card transactions
- **Fraudulent**: 492 fraudulent transactions (0.172%)
- **Features**: 28 PCA-transformed features + Time + Amount + Class

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Import project modules
import sys
sys.path.insert(0, str(Path.cwd().parent))

import config
from src.preprocessing import DataPreprocessor
from src.utils import setup_logging, print_data_statistics

logger = setup_logging('notebook')
print("Libraries imported successfully!")

## Step 2: Load Dataset

In [ ]:
# Load data
dataset_path = config.DATASET_PATH

if not dataset_path.exists():
    print(f"⚠️  Dataset not found at {dataset_path}")
    print("Please download from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud")
    print(f"And place creditcard.csv in {config.RAW_DATA_DIR}/")
else:
    df = pd.read_csv(dataset_path)
    print(f"✓ Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"\nFirst few rows:")
    df.head()

## Step 3: Data Overview

In [ ]:
# Basic information
print("Dataset Information:")
print(f"Total samples: {len(df):,}")
print(f"Total features: {df.shape[1]}")
print(f"\nData types:")
print(df.dtypes.value_counts())
print(f"\nMissing values:")
print(f"Total: {df.isnull().sum().sum()}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## Step 4: Target Variable Analysis

In [ ]:
# Class distribution
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print("Class Distribution:")
print(f"Legitimate (0): {class_counts[0]:,} ({class_pct[0]:.2f}%)")
print(f"Fraudulent (1): {class_counts[1]:,} ({class_pct[1]:.2f}%)")
print(f"\nClass Ratio (Fraud:Legitimate): 1:{class_counts[0]/class_counts[1]:.0f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(x='Class', data=df, ax=axes[0], palette=['green', 'red'])
axes[0].set_title('Transaction Count by Class')
axes[0].set_xticklabels(['Legitimate', 'Fraudulent'])

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[1].pie([class_counts[0], class_counts[1]], labels=['Legitimate', 'Fraudulent'],
           autopct='%1.2f%%', colors=colors, startangle=90)
axes[1].set_title('Class Distribution')

plt.tight_layout()
plt.show()
print("\n⚠️  HIGHLY IMBALANCED DATASET - requires special handling!")

## Step 5: Feature Statistics

In [ ]:
# Descriptive statistics for Amount
print("Amount Feature Statistics:")
print(df['Amount'].describe())
print(f"\nAmount Range: ${df['Amount'].min():.2f} - ${df['Amount'].max():.2f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(df['Amount'], bins=100, edgecolor='black', color='steelblue')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Frequency')

# Box plot by class
df.boxplot(column='Amount', by='Class', ax=axes[1]
axes[1].set_title('Amount Distribution by Class')
axes[1].set_xticklabels(['Legitimate', 'Fraudulent'])
axes[1].set_ylabel('Amount ($)')

plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.show()

# Compare statistics
print("\nAmount Statistics by Class:")
print(df.groupby('Class')['Amount'].describe())

## Step 6: Time Feature Analysis

In [ ]:
# Time analysis
print("Time Feature Statistics:")
print(f"Min time: {df['Time'].min()} seconds")
print(f"Max time: {df['Time'].max()} seconds")
print(f"Duration: {df['Time'].max() / 3600:.2f} hours")

# Create hour feature
df['Hour'] = (df['Time'] / 3600) % 24

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(df['Hour'], bins=24, edgecolor='black', color='steelblue')
axes[0].set_title('Transaction Time Distribution')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Frequency')

# By class
df[df['Class'] == 0]['Hour'].hist(bins=24, alpha=0.7, label='Legitimate', ax=axes[1])
df[df['Class'] == 1]['Hour'].hist(bins=24, alpha=0.7, label='Fraudulent', ax=axes[1])
axes[1].set_title('Transaction Time Distribution by Class')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 7: PCA Features Analysis

In [ ]:
# PCA features (V1-V28)
pca_features = [f'V{i}' for i in range(1, 29)]

print(f"PCA Features: {len(pca_features)} features")
print(f"Features: {pca_features[:5]} ... {pca_features[-5:]}")

# Statistics
print(f"\nPCA Features Statistics:")
print(df[pca_features].describe())

# Correlation with target
correlations = df[pca_features + ['Class']].corr()['Class'].drop('Class').abs().sort_values(ascending=False)
print(f"\nTop 10 Features by Correlation with Class:")
print(correlations.head(10))

# Visualization
plt.figure(figsize=(12, 6))
correlations.head(15).plot(kind='barh', color='steelblue')
plt.title('Top 15 Features by Absolute Correlation with Fraud')
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()

## Step 8: Correlation Analysis

In [ ]:
# Correlation matrix
plt.figure(figsize=(14, 10))

# Select top correlated features
top_features = correlations.head(10).index.tolist() + ['Amount', 'Class']
correlation_matrix = df[top_features].corr()

sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
           square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Top Features')
plt.tight_layout()
plt.show()

## Step 9: Anomaly Indicators

In [ ]:
# Analyze fraud patterns
fraud_df = df[df['Class'] == 1]
legit_df = df[df['Class'] == 0]

print("Fraud vs Legitimate - Amount Statistics:")
print(f"Fraudulent Avg: ${fraud_df['Amount'].mean():.2f}")
print(f"Legitimate Avg: ${legit_df['Amount'].mean():.2f}")
print(f"Fraudulent Median: ${fraud_df['Amount'].median():.2f}")
print(f"Legitimate Median: ${legit_df['Amount'].median():.2f}")

# Hour analysis
print(f"\nFraud vs Legitimate - Hour Statistics:")
print(f"Fraudulent Avg Hour: {fraud_df['Hour'].mean():.2f}")
print(f"Legitimate Avg Hour: {legit_df['Hour'].mean():.2f}")

# PCA feature analysis
print(f"\nPCA Feature Differences:")
for feature in correlations.head(5).index:
    fraud_mean = fraud_df[feature].mean()
    legit_mean = legit_df[feature].mean()
    print(f"{feature}: Fraud={fraud_mean:.4f}, Legit={legit_mean:.4f}, Diff={abs(fraud_mean-legit_mean):.4f}")

## Step 10: Key Insights

In [ ]:
print("🔍 KEY INSIGHTS FROM EXPLORATORY ANALYSIS")
print("="*60)

print("\n1. CLASS IMBALANCE")
print(f"   - Fraud rate: {class_pct[1]:.2f}%")
print(f"   - Ratio: 1 fraud for every {class_counts[0]/class_counts[1]:.0f} legitimate")
print("   ✓ Requires special handling (SMOTE, class weights, threshold tuning)")

print("\n2. FEATURE CHARACTERISTICS")
print(f"   - 28 PCA-transformed features (V1-V28)")
print(f"   - 1 Time feature (in seconds)")
print(f"   - 1 Amount feature (transaction amount in USD)")
print(f"   - No missing values detected")
print("   ✓ Data quality is high")

print("\n3. AMOUNT PATTERNS")
print(f"   - Average fraud amount: ${fraud_df['Amount'].mean():.2f}")
print(f"   - Average legitimate amount: ${legit_df['Amount'].mean():.2f}")
print(f"   - Frauds tend to be lower value transactions")
print("   ✓ Amount is a discriminative feature")

print("\n4. TIME PATTERNS")
print(f"   - Transactions span {df['Time'].max() / 3600:.1f} hours (~2 days)")
print(f"   - Consistent distribution throughout time period")
print("   ✓ No obvious temporal anomalies")

print("\n5. RECOMMENDED APPROACHES")
print("   ✓ Handle class imbalance with SMOTE")
print("   ✓ Use ensemble methods (Random Forest, XGBoost)")
print("   ✓ Focus on Precision-Recall metric (not accuracy)")
print("   ✓ Combine with anomaly detection methods")
print("   ✓ Optimize decision threshold for business requirements")
print("\n" + "="*60)